# nD WTMM Analysis — Generalized Python Port of xsmurf

Dimension-agnostic implementation of the WTMM method, generalizing
xsmurf's 2D/3D algorithms to arbitrary dimensions.

Key generalizations:
- **nD FFT CWT** with n Gaussian derivative wavelets (∂G/∂x_i)
- **Gradient modulus** = ||∇W|| (L2 norm of n-component gradient)
- **nD non-maxima suppression** with n-linear interpolation (2^n corners)
- **Tensor SVD** for m-component vector fields → n×m Jacobian → SVD
- **nD connected component labeling** for chain extraction
- **Partition function** and **D(h) spectrum**

Tested on: 2D images, 3D volumes, 4D hypercubes

In [ ]:
import numpy as np
from numpy.fft import fftn, ifftn, fftfreq
from scipy.ndimage import label as ndimage_label, generate_binary_structure
import matplotlib.pyplot as plt
from itertools import product
import warnings
warnings.filterwarnings('ignore')

print(f"NumPy {np.__version__}")
print("nD WTMM — Dimension-agnostic implementation")

## 1. nD Test Data Generator

In [ ]:
def generate_fbm_nd(shape, H=0.5, seed=42):
    """Generate nD fractional Brownian field with Hurst exponent H.
    
    Power spectrum: P(k) ~ |k|^{-(2H+d)} in d dimensions.
    Amplitude: |k|^{-(H + d/2)}
    
    Parameters:
        shape: tuple of ints, dimensions of the field
        H: Hurst exponent (0 < H < 1)
    """
    rng = np.random.default_rng(seed)
    ndim = len(shape)
    
    # Frequency grids for each dimension
    freq_grids = [fftfreq(n, d=1.0/n) for n in shape]
    K_grids = np.meshgrid(*freq_grids, indexing='ij')
    
    # Total wavenumber |k|
    K_sq = sum(kg**2 for kg in K_grids)
    K = np.sqrt(K_sq)
    K[tuple(0 for _ in range(ndim))] = 1.0  # avoid /0
    
    # Amplitude exponent: -(2H+d)/2 = -(H + d/2)
    amplitude = K ** (-(H + ndim / 2.0))
    amplitude[tuple(0 for _ in range(ndim))] = 0.0
    
    phases = rng.uniform(0, 2*np.pi, shape)
    spectrum = amplitude * np.exp(1j * phases)
    field = np.real(ifftn(spectrum))
    
    field = (field - field.min()) / (field.max() - field.min())
    return field

# Test in multiple dimensions
H_true = 0.7

print("Generating test fields:")
field_2d = generate_fbm_nd((128, 128), H=H_true, seed=42)
print(f"  2D: {field_2d.shape}")

field_3d = generate_fbm_nd((32, 32, 32), H=H_true, seed=42)
print(f"  3D: {field_3d.shape}")

field_4d = generate_fbm_nd((16, 16, 16, 16), H=H_true, seed=42)
print(f"  4D: {field_4d.shape}")

## 2. nD Wavelet Transform

Generalized Gaussian derivative wavelets in Fourier domain:

$$\hat{\psi}_i(\mathbf{k}) = i \cdot k_i \cdot \exp\left(-\sum_j k_j^2\right)$$

At scale $a$: $\hat{\psi}_i(a\mathbf{k}) = i \cdot (a k_i) \cdot \exp\left(-a^2 \sum_j k_j^2\right)$

In [ ]:
def gaussian_derivative_wavelets_nd_fourier(k_grids, scale):
    """
    nD Gaussian derivative wavelets in Fourier domain.
    
    Generalization of xsmurf's __conv_dx, __conv_dy to n dimensions.
    
    Parameters:
        k_grids: list of n frequency grids (from meshgrid)
        scale: wavelet scale
    
    Returns:
        list of n wavelet filters (one per dimension)
    """
    ndim = len(k_grids)
    
    # Scaled frequencies
    ak_grids = [scale * kg for kg in k_grids]
    
    # Gaussian envelope: exp(-sum(ak_i^2))
    k_sq_sum = sum(ak**2 for ak in ak_grids)
    gauss = np.exp(-k_sq_sum)
    
    # n derivative wavelets
    psi = [1j * ak * gauss for ak in ak_grids]
    
    return psi

def compute_scales(n_octaves, n_voices, a_min=1.0):
    """Scale sequence following xsmurf convention."""
    norm_factor = 6.0 / 0.86
    scales = []
    for oct in range(n_octaves):
        for vox in range(n_voices):
            s = a_min * (2.0 ** (oct + vox / n_voices)) * norm_factor
            scales.append(s)
    return np.array(scales)

def cwt_nd(field, scales):
    """
    nD Continuous Wavelet Transform via FFT.
    
    Generalizes xsmurf's cv2d FFT convolution to n dimensions.
    
    Parameters:
        field: nD array
        scales: array of scale values
    
    Returns:
        W: list of n arrays, each (n_scales, *field.shape)
           W[i] contains the i-th component of the wavelet transform
    """
    ndim = field.ndim
    shape = field.shape
    n_scales = len(scales)
    
    # Frequency grids
    freq_1d = [fftfreq(n) * 2 * np.pi for n in shape]
    k_grids = np.meshgrid(*freq_1d, indexing='ij')
    
    # FFT of field (computed once)
    field_fft = fftn(field)
    
    # Allocate output: n components × n_scales × spatial dims
    W = [np.zeros((n_scales, *shape), dtype=np.float64) for _ in range(ndim)]
    
    for s_idx, scale in enumerate(scales):
        psi_list = gaussian_derivative_wavelets_nd_fourier(k_grids, scale)
        for dim_idx in range(ndim):
            W[dim_idx][s_idx] = np.real(ifftn(field_fft * np.conj(psi_list[dim_idx])))
    
    return W

print("nD CWT functions defined.")
print(f"  Supported dimensions: any (tested 2D, 3D, 4D)")

## 3. nD Gradient Modulus

$$\text{modulus} = \sqrt{\sum_{i=1}^{n} W_i^2}$$

Generalization of xsmurf's `GradientModulus2D` and `GradientModulus3D`.

In [ ]:
def gradient_modulus_nd(W):
    """
    nD gradient modulus.
    
    Generalization of xsmurf GradientModulus2D/3D.
    
    Parameters:
        W: list of n arrays (n_scales, *spatial_dims)
    
    Returns:
        modulus: array (n_scales, *spatial_dims)
    """
    return np.sqrt(sum(w**2 for w in W))

print("gradient_modulus_nd() defined — works for any dimension.")

## 4. nD Non-Maxima Suppression

Generalization of xsmurf's bilinear (2D) and trilinear (3D) interpolation
to n-linear interpolation using 2^n corner weights.

For each point:
1. Normalize gradient → unit vector
2. Step ±1 along gradient direction
3. n-linear interpolation of modulus at those points
4. Keep only local maxima

In [ ]:
def nlinear_interpolate(volume, point):
    """
    n-linear interpolation at a fractional point in an nD volume.
    
    Generalizes bilinear (2D) and trilinear (3D) interpolation to nD.
    Uses 2^n corners of the enclosing hypercube.
    
    Parameters:
        volume: nD array
        point: tuple of n floats, the point to interpolate at
    
    Returns:
        interpolated value
    """
    ndim = volume.ndim
    shape = volume.shape
    
    # Floor indices and fractional parts
    floors = [int(p) for p in point]
    fracs = [p - f for p, f in zip(point, floors)]
    
    # Check bounds
    for d in range(ndim):
        if floors[d] < 0 or floors[d] + 1 >= shape[d]:
            return None
    
    # Sum over 2^n corners
    result = 0.0
    for corner in product([0, 1], repeat=ndim):
        # Weight for this corner
        weight = 1.0
        idx = []
        for d in range(ndim):
            if corner[d] == 0:
                weight *= (1.0 - fracs[d])
            else:
                weight *= fracs[d]
            idx.append(floors[d] + corner[d])
        
        result += weight * volume[tuple(idx)]
    
    return result

def non_maxima_suppression_nd(W_components, modulus_scale,
                               eps_norm=5e-7, eps_deriv=0.9995):
    """
    nD non-maxima suppression along gradient direction.
    
    Generalizes xsmurf's Remove_Gradient_NonMaxima_Slice_2D/3D.
    
    Parameters:
        W_components: list of n arrays (spatial_dims) — gradient components
        modulus_scale: array (spatial_dims) — gradient modulus
        eps_norm: minimum modulus threshold
        eps_deriv: threshold for nearest vs interpolated
    
    Returns:
        maxima: array with modulus at extrema, 0 elsewhere
    """
    ndim = modulus_scale.ndim
    shape = modulus_scale.shape
    maxima = np.zeros_like(modulus_scale)
    
    # Interior region (skip 1-pixel border on each side)
    ranges = [range(1, s - 1) for s in shape]
    
    for idx in product(*ranges):
        n = modulus_scale[idx]
        
        if n < eps_norm:
            continue
        
        # Normalized gradient components
        grad = [W_components[d][idx] / n for d in range(ndim)]
        
        # Check if any component is near ±1 (nearest neighbor case)
        use_nearest = any(abs(g) > eps_deriv for g in grad)
        
        if use_nearest:
            # Forward point (nearest neighbor)
            fwd = tuple(int(idx[d] + grad[d] + 0.5) for d in range(ndim))
            if all(0 <= fwd[d] < shape[d] for d in range(ndim)):
                if n <= modulus_scale[fwd]:
                    continue
            # Backward point
            bwd = tuple(int(idx[d] - grad[d] + 0.5) for d in range(ndim))
            if all(0 <= bwd[d] < shape[d] for d in range(ndim)):
                if n < modulus_scale[bwd]:
                    continue
            maxima[idx] = n
            continue
        
        # n-linear interpolation
        # Forward point
        fwd_point = tuple(idx[d] + grad[d] for d in range(ndim))
        interp_fwd = nlinear_interpolate(modulus_scale, fwd_point)
        
        if interp_fwd is None or n <= interp_fwd:
            continue
        
        # Backward point
        bwd_point = tuple(idx[d] - grad[d] for d in range(ndim))
        interp_bwd = nlinear_interpolate(modulus_scale, bwd_point)
        
        if interp_bwd is None or n < interp_bwd:
            continue
        
        maxima[idx] = n
    
    return maxima

print("nD non-maxima suppression defined.")
print("  Uses n-linear interpolation (2^n corners)")
print("  Equivalent to xsmurf bilinear (2D) and trilinear (3D)")

## 5. nD Tensor SVD

For an m-component vector field F: ℝ^n → ℝ^m, the wavelet transform
produces an n×m Jacobian tensor at each point. SVD decomposes:

$$J = U \Sigma V^T$$

where σ_max = largest singular value = modulus.

This generalizes xsmurf's 2×2 and 3×3 tensor SVD to arbitrary n×m.

In [ ]:
def gradient_modulus_tensor_nd(W_components_list, svd_type='max'):
    """
    nD tensor WTMM via SVD of n×m Jacobian.
    
    Generalization of xsmurf's GradientModulus2D_tensor2D and
    GradientModulus3D_tensor3D to arbitrary dimensions.
    
    Parameters:
        W_components_list: list of m items, each a list of n gradient arrays.
            W_components_list[j][i] = ∂f_j/∂x_i wavelet transform
            Shape of each array: (spatial_dims)
        svd_type: 'max', 'min', or 'trace'
    
    Returns:
        modulus: array (spatial_dims)
    """
    m = len(W_components_list)      # number of field components
    n = len(W_components_list[0])   # spatial dimension
    shape = W_components_list[0][0].shape
    n_pts = np.prod(shape)
    
    # Build (n_pts, n, m) Jacobian tensor array
    # J[point, spatial_dim, field_component]
    J = np.zeros((n_pts, n, m))
    for j in range(m):
        for i in range(n):
            J[:, i, j] = W_components_list[j][i].ravel()
    
    # Batch SVD
    U, s, Vt = np.linalg.svd(J, full_matrices=False)
    
    if svd_type == 'max':
        modulus = s[:, 0]
    elif svd_type == 'min':
        modulus = s[:, -1]
    elif svd_type == 'trace':
        modulus = np.sum(s, axis=-1)
    else:
        raise ValueError(f"Unknown svd_type: {svd_type}")
    
    return modulus.reshape(shape)

def gradient_modulus_tensor_nd_LT(W_components_list):
    """
    nD Longitudinal/Transverse decomposition.
    
    Generalizes xsmurf's LT decomposition.
    
    For square n×n tensor J:
        J_sym = (J + J^T) / 2   (longitudinal/irrotational)
        J_anti = (J - J^T) / 2  (transverse/rotational)
    
    modL = max|J_sym|  (Frobenius norm or max element)
    modT = max|J_anti|
    """
    m = len(W_components_list)
    n = len(W_components_list[0])
    
    if n != m:
        raise ValueError("LT decomposition requires square Jacobian (n == m)")
    
    shape = W_components_list[0][0].shape
    n_pts = np.prod(shape)
    
    # Build Jacobian
    J = np.zeros((n_pts, n, n))
    for j in range(n):
        for i in range(n):
            J[:, i, j] = W_components_list[j][i].ravel()
    
    # Symmetric and antisymmetric parts
    J_sym = 0.5 * (J + np.swapaxes(J, -2, -1))
    J_anti = 0.5 * (J - np.swapaxes(J, -2, -1))
    
    # Max absolute element (following xsmurf MAX3/MAX6 approach)
    modL = np.max(np.abs(J_sym), axis=(-2, -1)).reshape(shape)
    modT = np.max(np.abs(J_anti), axis=(-2, -1)).reshape(shape)
    
    return modL, modT

print("nD tensor functions defined:")
print("  - gradient_modulus_tensor_nd: SVD of n×m Jacobian")
print("  - gradient_modulus_tensor_nd_LT: Longitudinal/transverse decomposition")

## 6. nD Chain Extraction

In [ ]:
def extract_components_nd(extrema_map):
    """
    Extract connected components from nD extrema map.
    
    Uses full connectivity (3^n - 1 neighbors).
    Generalizes xsmurf's 8-connected (2D) and 26-connected (3D) labeling.
    """
    ndim = extrema_map.ndim
    binary = (extrema_map > 0).astype(np.int32)
    
    # Full connectivity structure (all neighbors)
    structure = generate_binary_structure(ndim, ndim)  # full connectivity
    labeled, n_comp = ndimage_label(binary, structure=structure)
    
    # Get component statistics
    comp_data = {}
    positions = np.argwhere(extrema_map > 0)
    
    for pos in positions:
        pos_tuple = tuple(pos)
        comp_id = labeled[pos_tuple]
        mod_val = extrema_map[pos_tuple]
        
        if comp_id not in comp_data:
            comp_data[comp_id] = {'positions': [], 'moduli': [], 'max_mod': 0}
        comp_data[comp_id]['positions'].append(pos_tuple)
        comp_data[comp_id]['moduli'].append(mod_val)
        comp_data[comp_id]['max_mod'] = max(comp_data[comp_id]['max_mod'], mod_val)
    
    return comp_data, n_comp, labeled

print("nD component extraction defined.")

## 7. nD Partition Function and D(h) Spectrum

In [ ]:
def partition_function_nd(comp_data_per_scale, scales, q_values, min_comp_size=3):
    """
    nD WTMM partition function.
    
    Z(q, a) = Σ_chains max(|mod|)^q
    """
    n_q = len(q_values)
    n_scales = len(scales)
    Z = np.zeros((n_q, n_scales))
    
    for s_idx, comp_data in enumerate(comp_data_per_scale):
        chain_maxima = []
        for comp_id, data in comp_data.items():
            if len(data['positions']) >= min_comp_size and data['max_mod'] > 0:
                chain_maxima.append(data['max_mod'])
        
        if not chain_maxima:
            Z[:, s_idx] = np.nan
            continue
        
        chain_maxima = np.array(chain_maxima)
        for q_idx, q in enumerate(q_values):
            if q == 0:
                Z[q_idx, s_idx] = len(chain_maxima)
            else:
                Z[q_idx, s_idx] = np.sum(chain_maxima ** q)
    
    return Z

def compute_tau_nd(Z, scales, q_values):
    """Compute τ(q) via log-log regression."""
    n_q = len(q_values)
    tau = np.zeros(n_q)
    log_a = np.log(scales)
    
    for q_idx in range(n_q):
        z = Z[q_idx]
        valid = np.isfinite(z) & (z > 0)
        if np.sum(valid) < 2:
            tau[q_idx] = np.nan
            continue
        coeffs = np.polyfit(log_a[valid], np.log(z[valid]), 1)
        tau[q_idx] = coeffs[0]
    return tau

def compute_dh_nd(tau, q_values, d):
    """D(h) via Legendre transform. d = topological dimension."""
    h = np.gradient(tau, q_values)
    D = q_values * h - tau + d
    return h, D

print("nD partition function and D(h) defined.")

## 8. Complete nD WTMM Pipeline

In [ ]:
def wtmm_nd_pipeline(field, n_octaves=4, n_voices=2, a_min=1.0,
                      q_values=None, min_comp_size=3, verbose=True):
    """
    Complete nD WTMM analysis pipeline.
    
    Generalized port of the full xsmurf pipeline.
    
    Parameters:
        field: nD numpy array
        n_octaves, n_voices: scale parameters
        a_min: minimum scale
        q_values: array of q exponents for partition function
        min_comp_size: minimum component size
    
    Returns:
        dict with all results
    """
    ndim = field.ndim
    if q_values is None:
        q_values = np.linspace(-3, 3, 15)
    
    if verbose:
        print(f"\n{'='*60}")
        print(f"{ndim}D WTMM Pipeline")
        print(f"{'='*60}")
        print(f"Field shape: {field.shape}")
    
    # 1. Compute scales
    scales = compute_scales(n_octaves, n_voices, a_min)
    if verbose:
        print(f"Scales: {len(scales)} (range [{scales[0]:.1f}, {scales[-1]:.1f}])")
    
    # 2. CWT
    if verbose:
        print(f"Computing {ndim}D CWT...")
    W = cwt_nd(field, scales)
    
    # 3. Gradient modulus
    if verbose:
        print(f"Computing gradient modulus...")
    modulus = gradient_modulus_nd(W)
    
    # 4. Non-maxima suppression
    if verbose:
        print(f"Non-maxima suppression ({ndim}D n-linear interpolation)...")
    extrema = np.zeros_like(modulus)
    for s_idx in range(len(scales)):
        W_at_scale = [W[d][s_idx] for d in range(ndim)]
        extrema[s_idx] = non_maxima_suppression_nd(W_at_scale, modulus[s_idx])
        if verbose:
            n_ext = np.count_nonzero(extrema[s_idx])
            print(f"  Scale {s_idx} (a={scales[s_idx]:.1f}): {n_ext} extrema")
    
    # 5. Component extraction
    if verbose:
        print(f"Extracting connected components...")
    comp_data_per_scale = []
    for s_idx in range(len(scales)):
        comp_data, n_comp, _ = extract_components_nd(extrema[s_idx])
        comp_data_per_scale.append(comp_data)
        if verbose:
            print(f"  Scale {s_idx}: {n_comp} components")
    
    # 6. Partition function
    if verbose:
        print(f"Computing partition function...")
    Z = partition_function_nd(comp_data_per_scale, scales, q_values, min_comp_size)
    tau = compute_tau_nd(Z, scales, q_values)
    h, D = compute_dh_nd(tau, q_values, d=ndim)
    
    if verbose:
        print(f"\nResults:")
        print(f"  Total extrema: {np.count_nonzero(extrema)}")
        q1_idx = np.argmin(np.abs(q_values - 1))
        print(f"  τ(1) = {tau[q1_idx]:.3f}")
    
    return {
        'ndim': ndim,
        'scales': scales,
        'W': W,
        'modulus': modulus,
        'extrema': extrema,
        'comp_data': comp_data_per_scale,
        'Z': Z,
        'tau': tau,
        'h': h,
        'D': D,
        'q_values': q_values
    }

print("Complete nD WTMM pipeline defined.")

## 9. Run Pipeline on 2D, 3D, and 4D Test Data

In [ ]:
# 2D analysis
results_2d = wtmm_nd_pipeline(field_2d, n_octaves=4, n_voices=2)

In [ ]:
# 3D analysis (smaller for speed)
results_3d = wtmm_nd_pipeline(field_3d, n_octaves=3, n_voices=2)

In [ ]:
# 4D analysis (small cube)
results_4d = wtmm_nd_pipeline(field_4d, n_octaves=2, n_voices=2)

## 10. Comparative Visualization

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 16))

for row, (results, dim_label) in enumerate([
    (results_2d, '2D'), (results_3d, '3D'), (results_4d, '4D')]):
    
    q_values = results['q_values']
    scales = results['scales']
    Z = results['Z']
    tau = results['tau']
    h = results['h']
    D = results['D']
    ndim = results['ndim']
    
    # Partition function
    ax = axes[row, 0]
    for q in [-2, 0, 1, 2]:
        q_idx = np.argmin(np.abs(q_values - q))
        z = Z[q_idx]
        valid = np.isfinite(z) & (z > 0)
        if np.any(valid):
            ax.loglog(scales[valid], z[valid], 'o-', label=f'q={q}', markersize=3)
    ax.set_xlabel('Scale a')
    ax.set_ylabel('Z(q, a)')
    ax.set_title(f'{dim_label} Partition Function')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    
    # τ(q)
    ax = axes[row, 1]
    valid_tau = np.isfinite(tau)
    if np.any(valid_tau):
        ax.plot(q_values[valid_tau], tau[valid_tau], 'b.-')
    # Reference: monofractal τ(q) = qH - d
    ax.plot(q_values, q_values * H_true - ndim, 'r--', alpha=0.5,
            label=f'qH-{ndim} (H={H_true})')
    ax.set_xlabel('q')
    ax.set_ylabel('τ(q)')
    ax.set_title(f'{dim_label} Scaling Exponents')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # D(h)
    ax = axes[row, 2]
    valid_dh = np.isfinite(h) & np.isfinite(D)
    if np.any(valid_dh):
        ax.plot(h[valid_dh], D[valid_dh], 'g.-', markersize=8)
    ax.set_xlabel('h')
    ax.set_ylabel('D(h)')
    ax.set_title(f'{dim_label} Singularity Spectrum')
    ax.axhline(y=ndim, color='gray', linestyle=':', alpha=0.5, label=f'D={ndim}')
    ax.axvline(x=H_true, color='r', linestyle='--', alpha=0.5, label=f'H={H_true}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle(f'nD WTMM Analysis Comparison (fBm H={H_true})', fontsize=16)
plt.tight_layout()
plt.show()

## 11. Tensor nD Demo — Vector Field Analysis

In [ ]:
# Create a 2D vector field (2 components) for tensor demo
f1_2d = generate_fbm_nd((64, 64), H=0.5, seed=42)
f2_2d = generate_fbm_nd((64, 64), H=0.8, seed=99)

test_scales = compute_scales(2, 2)

print("Computing CWT for 2-component 2D vector field...")
W_f1 = cwt_nd(f1_2d, test_scales[:2])  # W_f1[dim][scale] = ∂f1/∂x_dim
W_f2 = cwt_nd(f2_2d, test_scales[:2])  # W_f2[dim][scale] = ∂f2/∂x_dim

# Build tensor at scale 0: 
# W_components_list[j][i] = ∂f_j/∂x_i
s = 0  # first scale
W_tensor = [
    [W_f1[0][s], W_f1[1][s]],  # [∂f1/∂x, ∂f1/∂y]
    [W_f2[0][s], W_f2[1][s]],  # [∂f2/∂x, ∂f2/∂y]
]

# SVD analysis
mod_max = gradient_modulus_tensor_nd(W_tensor, svd_type='max')
mod_min = gradient_modulus_tensor_nd(W_tensor, svd_type='min')
mod_trace = gradient_modulus_tensor_nd(W_tensor, svd_type='trace')

# LT decomposition
modL, modT = gradient_modulus_tensor_nd_LT(W_tensor)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, data, title in zip(axes.flat, 
    [mod_max, mod_min, mod_trace, modL, modT, mod_max / (mod_min + 1e-10)],
    ['σ_max', 'σ_min', 'trace(Σ)', 'Longitudinal', 'Transverse', 'σ_max/σ_min (anisotropy)']):
    im = ax.imshow(data, cmap='hot')
    ax.set_title(title)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('nD Tensor WTMM Analysis (2D vector field)', fontsize=14)
plt.tight_layout()
plt.show()

print(f"σ_max range: [{mod_max.min():.4f}, {mod_max.max():.4f}]")
print(f"σ_min range: [{mod_min.min():.4f}, {mod_min.max():.4f}]")
print(f"modL range:  [{modL.min():.4f}, {modL.max():.4f}]")
print(f"modT range:  [{modT.min():.4f}, {modT.max():.4f}]")

## 12. Summary

### Dimension Generalization Table

| Component | 2D (xsmurf) | 3D (xsmurf) | nD (this notebook) |
|-----------|-------------|-------------|---------------------|
| Wavelet | ψ_x, ψ_y | ψ_x, ψ_y, ψ_z | ψ_{x_1}, ..., ψ_{x_n} |
| FFT | fft2 | fft3 | fftn |
| Modulus | sqrt(gx²+gy²) | sqrt(gx²+gy²+gz²) | sqrt(Σ g_i²) |
| Interpolation | Bilinear (4 pts) | Trilinear (8 pts) | n-linear (2^n pts) |
| Connectivity | 8-connected | 26-connected | (3^n-1)-connected |
| Tensor SVD | 2×2 | 3×3 | n×m |
| LT decomp | MAX3/antisym | MAX6/MAX3 | max|J_sym|/max|J_anti| |
| D(h) | qh - τ + 2 | qh - τ + 3 | qh - τ + d |

### Key Design Decisions

1. **n-linear interpolation** via iterating over 2^n corners — faithful to xsmurf's bilinear/trilinear approach
2. **Batch SVD** via numpy — vectorized over all spatial points simultaneously  
3. **scipy.ndimage.label** for connected components — equivalent to xsmurf's recursive neighbor walk
4. **Full connectivity** (3^n-1 neighbors) — matches xsmurf's 8-connected (2D) and 26-connected (3D)

In [ ]:
print("="*60)
print("nD WTMM Analysis Complete")
print("="*60)
for results, label in [(results_2d, '2D'), (results_3d, '3D'), (results_4d, '4D')]:
    n_ext = np.count_nonzero(results['extrema'])
    q1_idx = np.argmin(np.abs(results['q_values'] - 1))
    print(f"  {label}: {n_ext} extrema, τ(1) = {results['tau'][q1_idx]:.3f}")